In [1]:
# Uncomment kode di bawah ini jika terdapat masalah dengan Java di Windows

import os
import sys

JAVA = r"C:\Program Files\jdk-17.0.12" # Ganti dengan path JDK yang sesuai

os.environ["JAVA_HOME"] = JAVA
os.environ["PATH"] = os.path.join(JAVA, "bin") + os.path.pathsep + os.environ.get("PATH", "")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [2]:
%pip install -r requirements.txt
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, trim, lower

from pyspark.ml import Pipeline
from pyspark.ml.classification import LinearSVC
from pyspark.ml.feature import (
    IDF,
    HashingTF,
    Normalizer,
    StringIndexer,
    Tokenizer,
)

spark = SparkSession.builder \
    .appName("AmazonReviewsPolarity") \
    .config("spark.driver.memory", "4g") \
    .master("local[*]") \
    .getOrCreate()

df_train = spark.read.csv("../datasets/train.csv", header=True, inferSchema=True)
df_train = df_train.toDF("label", "title", "text")

df_test = spark.read.csv("../datasets/test.csv", header=True, inferSchema=True)
df_test = df_test.toDF("label", "title", "text")

ERROR: Cannot install h11==0.16.0 and httpcore==1.0.7 because these package versions have conflicting dependencies.

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


Note: you may need to restart the kernel to use updated packages.Defaulting to user installation because normal site-packages is not writeable
INFO: pip is looking at multiple versions of httpcore to determine which version is compatible with other requirements. This could take a while.

The conflict is caused by:
    The user requested h11==0.16.0
    httpcore 1.0.7 depends on h11<0.15 and >=0.13

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict




In [ ]:
def preprocessing(df):
    df_clean = df.dropna()
    
    df_clean = df_clean.dropDuplicates(["title", "text"])
    
    df_clean = df_clean \
        .withColumn("title", trim(col("title"))) \
        .withColumn("text", trim(col("text")))
    
    invalid_words = ["n/a", "null", "undefined", "-"]
    
    df_clean = df_clean.filter(
        col("label").isin(1, 2) &
        (length(col("title")) >= 3) &
        (length(col("text")) >= 3) &
        (lower(col("text")).isin(invalid_words))
    )
    
    return df_clean

In [ ]:
tokenizer = Tokenizer(inputCol="text", outputCol="words")

hashingTF = HashingTF(
    inputCol="words", outputCol="rawFeatures", numFeatures=10000
)

idf = IDF(inputCol="rawFeatures", outputCol="idfFeatures")

normalizer = Normalizer(inputCol="idfFeatures", outputCol="features", p=2.0)

label_indexer = StringIndexer(inputCol="label", outputCol="label_index")

svm = LinearSVC(featuresCol="features", labelCol="label_index", maxIter=15)

pipeline = Pipeline(
    stages=[tokenizer, hashingTF, idf, normalizer, label_indexer, svm]
)

df_train = preprocessing(df_train)
model = pipeline.fit(df_train)

df_test = preprocessing(df_test)
predictions = model.transform(df_test)

predictions.select("text", "label_index", "prediction").show(100, truncate=30)